# LLM-Orchestrated Attrition Pipeline — Function Calling
The LLM autonomously orchestrates the full pipeline using OpenAI function calling. It decides when to run each script, checks outputs, interprets results using REFERENCE.md, and provides HR recommendations.


## Setup


In [11]:
import os
import sys
import json
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully")

def read_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

skill_md = read_file("SKILL.md")
reference_md = read_file("REFERENCE.md")
print(f"SKILL.md: {len(skill_md)} chars | REFERENCE.md: {len(reference_md)} chars")


OpenAI client initialized successfully
SKILL.md: 9665 chars | REFERENCE.md: 9900 chars


## Tool Definitions (Functions the LLM Can Call)


In [12]:
# Define the 4 tools the LLM can call
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_validation",
            "description": (
                "Run Stage 1: Data validation and profiling on the input CSV. "
                "Validates schema, nulls, duplicates, class balance, ordinal ranges. "
                "Returns validation report as JSON string. "
                "Always run this first before any other stage."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "input_path": {
                        "type": "string",
                        "description": "Path to input CSV file"
                    },
                    "output_path": {
                        "type": "string",
                        "description": "Path to save validation report JSON",
                        "default": "outputs/validation_report.json"
                    }
                },
                "required": ["input_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_feature_engineering",
            "description": (
                "Run Stage 2: Feature engineering on raw CSV. "
                "Encodes target, binary columns, ordinal columns, one-hot encodes "
                "categoricals, and creates 8 domain-specific engineered features. "
                "Only call this after validation has passed."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "input_path": {
                        "type": "string",
                        "description": "Path to raw input CSV"
                    },
                    "output_path": {
                        "type": "string",
                        "description": "Path to save features CSV",
                        "default": "outputs/features.csv"
                    },
                    "manifest_path": {
                        "type": "string",
                        "description": "Path to save feature manifest JSON",
                        "default": "outputs/feature_manifest.json"
                    }
                },
                "required": ["input_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_model_training",
            "description": (
                "Run Stage 3: Train and evaluate 3 classification models "
                "(Logistic Regression, Random Forest, Gradient Boosting) "
                "with SMOTE balancing and 5-fold cross validation. "
                "Computes SHAP feature importances and risk segmentation. "
                "Only call this after feature engineering has completed."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "input_path": {
                        "type": "string",
                        "description": "Path to engineered features CSV",
                        "default": "outputs/features.csv"
                    },
                    "output_path": {
                        "type": "string",
                        "description": "Path to save model results JSON",
                        "default": "outputs/model_results.json"
                    },
                    "random_seed": {
                        "type": "integer",
                        "description": "Random seed for reproducibility",
                        "default": 42
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_report_generation",
            "description": (
                "Run Stage 4: Generate a self-contained HTML analytics report "
                "with charts, model comparison, SHAP importances, risk segments, "
                "and business recommendations. "
                "Only call this after model training has completed."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "features_path": {
                        "type": "string",
                        "description": "Path to features CSV",
                        "default": "outputs/features.csv"
                    },
                    "model_results_path": {
                        "type": "string",
                        "description": "Path to model results JSON",
                        "default": "outputs/model_results.json"
                    },
                    "output_path": {
                        "type": "string",
                        "description": "Path to save HTML report",
                        "default": "outputs/attrition_report.html"
                    },
                    "company_name": {
                        "type": "string",
                        "description": "Company name label for report",
                        "default": "IBM HR Analytics"
                    }
                },
                "required": []
            }
        }
    }
]

print(f"Defined {len(TOOLS)} tools for LLM:")
for t in TOOLS:
    print(f"  - {t['function']['name']}: {t['function']['description'][:60]}...")


Defined 4 tools for LLM:
  - run_validation: Run Stage 1: Data validation and profiling on the input CSV....
  - run_feature_engineering: Run Stage 2: Feature engineering on raw CSV. Encodes target,...
  - run_model_training: Run Stage 3: Train and evaluate 3 classification models (Log...
  - run_report_generation: Run Stage 4: Generate a self-contained HTML analytics report...


## Tool Executor (Python runs what LLM decides)


In [13]:
def execute_tool(tool_name: str, tool_args: dict) -> str:
    """Execute whichever tool the LLM decided to call.
    
    Returns the script output as a string to feed back to LLM.
    """
    os.makedirs("outputs", exist_ok=True)
    
    print(f"\n>>> LLM called: {tool_name}({json.dumps(tool_args, indent=2)})")
    print("-" * 50)

    if tool_name == "run_validation":
        cmd = [
            sys.executable, "scripts/validate_data.py",
            "--input", tool_args.get("input_path"),
            "--output", tool_args.get("output_path", "outputs/validation_report.json")
        ]

    elif tool_name == "run_feature_engineering":
        cmd = [
            sys.executable, "scripts/feature_engineering.py",
            "--input", tool_args.get("input_path"),
            "--output", tool_args.get("output_path", "outputs/features.csv"),
            "--manifest", tool_args.get("manifest_path", "outputs/feature_manifest.json")
        ]

    elif tool_name == "run_model_training":
        cmd = [
            sys.executable, "scripts/run_models.py",
            "--input", tool_args.get("input_path", "outputs/features.csv"),
            "--output", tool_args.get("output_path", "outputs/model_results.json"),
            "--random-seed", str(tool_args.get("random_seed", 42))
        ]

    elif tool_name == "run_report_generation":
        cmd = [
            sys.executable, "scripts/generate_report.py",
            "--features", tool_args.get("features_path", "outputs/features.csv"),
            "--model-results", tool_args.get("model_results_path", "outputs/model_results.json"),
            "--output", tool_args.get("output_path", "outputs/attrition_report.html"),
            "--company-name", tool_args.get("company_name", "IBM HR Analytics")
        ]

    else:
        return f"ERROR: Unknown tool '{tool_name}'"

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stdout
    if result.stderr:
        output += f"\nSTDERR:\n{result.stderr}"
    
    print(output)
    print(f">>> Return code: {result.returncode}")
    return output


def save_trace(run_name: str, messages: list):
    """Save full conversation + tool calls as execution evidence."""
    path = f"outputs/{run_name}_trace.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({
            "run_name": run_name,
            "model": "gpt-4o",
            "total_messages": len(messages),
            "messages": messages
        }, f, indent=2)
    print(f"\nTrace saved: {path}")


## Agentic Loop (Core Engine)


In [14]:
def run_agentic_pipeline(
    user_prompt: str,
    run_name: str,
    max_iterations: int = 20
) -> list:
    """
    Core agentic loop. LLM decides which tools to call,
    Python executes them, results fed back to LLM.
    Continues until LLM stops calling tools.
    
    Returns full message history as execution trace.
    """
    
    SYSTEM_PROMPT = f"""
You are an expert HR analytics consultant executing an attrition 
analysis skill package. You have 4 tools available corresponding 
to the 4 pipeline stages in SKILL.md.

Your job:
1. Read the user's request carefully
2. Execute the pipeline stages IN ORDER using your tools:
   run_validation → run_feature_engineering → 
   run_model_training → run_report_generation
3. After EACH tool call, interpret the output using REFERENCE.md
4. If validation FAILS (return code non-zero or validation_passed=false):
   STOP and explain what the user needs to fix
5. After all stages complete, provide a final executive summary with:
   - Overall attrition risk assessment
   - Top 3 HR interventions
   - Which employee segments need immediate attention

Always structure your text responses as:
STAGE STATUS: PASSED / FAILED / WARNING
KEY FINDINGS: (bullet points)
BUSINESS INTERPRETATION: (plain language HR meaning)
NEXT ACTION: (what you will do next)

SKILL.md:
{skill_md}

REFERENCE.md:
{reference_md}
"""

    messages = [
        {"role": "user", "content": user_prompt}
    ]
    
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n{'='*60}")
        print(f"ITERATION {iteration}")
        print('='*60)
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT}
            ] + messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
            max_tokens=2000
        )
        
        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        
        print(f"Finish reason: {finish_reason}")
        
        if msg.content:
            print(f"\nLLM: {msg.content}")
        
        assistant_msg = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)
        
        if finish_reason == "stop" or not msg.tool_calls:
            print("\nLLM finished — no more tool calls.")
            break
        
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            tool_output = execute_tool(tool_name, tool_args)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_output
            })
    
    save_trace(run_name, messages)
    return messages

print("Agentic loop ready.")


Agentic loop ready.


## Run 1 — Happy Path (Full IBM Dataset)


In [15]:
print("STARTING RUN 1 — HAPPY PATH")
print("Dataset: data/employee-attrition.csv (1470 rows)")
print("The LLM will autonomously run all 4 pipeline stages.\n")

messages_run1 = run_agentic_pipeline(
    user_prompt="""
Please run the complete employee attrition analysis pipeline 
on the IBM HR dataset at: data/employee-attrition.csv

Use company name "IBM HR Analytics" for the report.
Use random seed 42 for reproducibility.

Execute all 4 stages in order, interpret each result using 
REFERENCE.md, and provide a final executive summary with 
top HR interventions at the end.
""",
    run_name="run1_happy_path"
)

print(f"\nRun 1 complete. Messages in trace: {len(messages_run1)}")


STARTING RUN 1 — HAPPY PATH
Dataset: data/employee-attrition.csv (1470 rows)
The LLM will autonomously run all 4 pipeline stages.


ITERATION 1
Finish reason: tool_calls

>>> LLM called: run_validation({
  "input_path": "data/employee-attrition.csv"
})
--------------------------------------------------

=== Employee Attrition Data Validation Summary ===
✅ Basic checks: rows=1470, cols_after_drop=31, dropped=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
✅ Null checks: analyzed all columns
✅ Target analysis: Yes=237, No=1233, Ratio=5.202532
✅ Numeric profiling: 23 columns
✅ Categorical profiling: 8 columns
✅ Duplicate check: 0 duplicate row(s)
✅ Ordinal range validation completed

Warnings:
- Target class imbalance ratio is high: 5.2025 (>5.0).

Final verdict: PASSED

>>> Return code: 0

ITERATION 2
Finish reason: tool_calls

LLM: STAGE STATUS: PASSED
KEY FINDINGS:
- The dataset contains 1470 rows and 31 columns after dropping constants.
- No missing values exceed the th

## Run 2 — Bad Data Scenario


In [16]:
# Create bad dataset
df = pd.read_csv("data/employee-attrition.csv")
bad_df = df.drop(columns=["Attrition", "MonthlyIncome", "OverTime"])
bad_df.to_csv("outputs/bad_data_test.csv", index=False)
print(f"Bad dataset: {bad_df.shape} — missing Attrition, MonthlyIncome, OverTime")

print("\nSTARTING RUN 2 — BAD DATA SCENARIO")
print("LLM should detect failure and stop with clear explanation.\n")

messages_run2 = run_agentic_pipeline(
    user_prompt="""
Please run the employee attrition analysis pipeline on this dataset:
outputs/bad_data_test.csv

Execute the pipeline following SKILL.md. If any stage fails,
explain clearly what is wrong and what the user needs to fix.
""",
    run_name="run2_bad_data"
)

print(f"\nRun 2 complete. Messages in trace: {len(messages_run2)}")


Bad dataset: (1470, 32) — missing Attrition, MonthlyIncome, OverTime

STARTING RUN 2 — BAD DATA SCENARIO
LLM should detect failure and stop with clear explanation.


ITERATION 1
Finish reason: tool_calls

>>> LLM called: run_validation({
  "input_path": "outputs/bad_data_test.csv"
})
--------------------------------------------------

=== Employee Attrition Data Validation Summary ===
✅ Basic checks: rows=1470, cols_after_drop=28, dropped=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
✅ Null checks: analyzed all columns
✅ Target analysis: Yes=0, No=0, Ratio=None
✅ Numeric profiling: 22 columns
✅ Categorical profiling: 6 columns
✅ Duplicate check: 0 duplicate row(s)
✅ Ordinal range validation completed

Warnings: None

Errors:
- Missing required columns after drop step -> target: ['Attrition']; categorical: ['OverTime']; numeric: ['MonthlyIncome']
- Cannot analyze target: 'Attrition' column missing.

Final verdict: FAILED - Missing required columns after drop step -> tar

## Run 3 — Junior Employees Subset


In [17]:
# Create junior subset
df = pd.read_csv("data/employee-attrition.csv")
junior_df = df[df["JobLevel"] <= 2].copy()
junior_df.to_csv("outputs/junior_employees.csv", index=False)
print(f"Junior subset: {junior_df.shape}")
print(f"Attrition rate: {(junior_df['Attrition']=='Yes').mean()*100:.1f}%")

print("\nSTARTING RUN 3 — JUNIOR EMPLOYEES SUBSET")
print("LLM will run full pipeline on subset and compare to Run 1.\n")

messages_run3 = run_agentic_pipeline(
    user_prompt="""
Please run the complete employee attrition analysis pipeline 
on the junior employees subset at: outputs/junior_employees.csv

Use company name "IBM HR Analytics — Junior Employees" for the report.
Use random seed 42.

IMPORTANT: Use dedicated output files so you do not overwrite the full-dataset run:
- After validation, call run_feature_engineering with:
  output_path=outputs/junior_features.csv and manifest_path=outputs/junior_manifest.json
- Call run_model_training with:
  input_path=outputs/junior_features.csv and output_path=outputs/junior_model_results.json
- Call run_report_generation with:
  features_path=outputs/junior_features.csv, model_results_path=outputs/junior_model_results.json,
  output_path=outputs/junior_report.html, company_name="IBM HR Analytics — Junior Employees"

After completing all stages, compare your findings to a typical 
full-workforce analysis and provide recommendations specific to 
junior/entry-level employee retention.
""",
    run_name="run3_junior_subset"
)

print(f"\nRun 3 complete. Messages in trace: {len(messages_run3)}")


Junior subset: (1077, 35)
Attrition rate: 18.1%

STARTING RUN 3 — JUNIOR EMPLOYEES SUBSET
LLM will run full pipeline on subset and compare to Run 1.


ITERATION 1
Finish reason: tool_calls

>>> LLM called: run_validation({
  "input_path": "outputs/junior_employees.csv",
  "output_path": "outputs/junior_validation_report.json"
})
--------------------------------------------------

=== Employee Attrition Data Validation Summary ===
✅ Basic checks: rows=1077, cols_after_drop=31, dropped=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
✅ Null checks: analyzed all columns
✅ Target analysis: Yes=195, No=882, Ratio=4.523077
✅ Numeric profiling: 23 columns
✅ Categorical profiling: 8 columns
✅ Duplicate check: 0 duplicate row(s)
✅ Ordinal range validation completed

Warnings:
- Numeric column 'YearsSinceLastPromotion' is highly skewed (skewness=2.1282); consider log transform.

Final verdict: PASSED

>>> Return code: 0

ITERATION 2
Finish reason: tool_calls

LLM: STAGE STATUS: PAS

## Evidence Summary


In [18]:
print("\n" + "="*60)
print("EXECUTION EVIDENCE SUMMARY")
print("="*60)

evidence = {
    "Run 1 Trace (LLM conversation)": "outputs/run1_happy_path_trace.json",
    "Run 1 HTML Report":              "outputs/attrition_report.html",
    "Run 2 Trace (bad data)":         "outputs/run2_bad_data_trace.json",
    "Run 3 Trace (junior subset)":    "outputs/run3_junior_subset_trace.json",
    "Run 3 HTML Report":              "outputs/junior_report.html",
    "Validation Report":              "outputs/validation_report.json",
    "Model Results":                  "outputs/model_results.json",
    "Feature Manifest":               "outputs/feature_manifest.json",
}

all_present = True
for name, path in evidence.items():
    exists = os.path.exists(path)
    if not exists:
        all_present = False
    size = f"{os.path.getsize(path)/1024:.1f} KB" if exists else "MISSING"
    print(f"{'✅' if exists else '❌'} {name}: {size}")

print(f"\n{'✅ All evidence present' if all_present else '❌ Some files missing'}")
print("\nSubmit outputs/ folder + GitHub repo as assignment deliverables.")



EXECUTION EVIDENCE SUMMARY
✅ Run 1 Trace (LLM conversation): 8.8 KB
✅ Run 1 HTML Report: 398.1 KB
✅ Run 2 Trace (bad data): 2.6 KB
✅ Run 3 Trace (junior subset): 10.4 KB
✅ Run 3 HTML Report: 399.2 KB
✅ Validation Report: 9.9 KB
✅ Model Results: 4.1 KB
✅ Feature Manifest: 4.3 KB

✅ All evidence present

Submit outputs/ folder + GitHub repo as assignment deliverables.
